In [1]:
!pip -q install evaluate jiwer soundfile librosa torchcodec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 40.9 MB/s eta 0:00:0000:01


In [2]:
import torch
import evaluate
import numpy as np
from torch.utils.data import DataLoader
from datasets import load_dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration, set_seed

set_seed(42)

### Load raw LibriSpeech splits

In [ ]:
train_raw = load_dataset("openslr/librispeech_asr", "clean", split="train.100[:1000]")
val_raw = load_dataset("openslr/librispeech_asr", "clean", split="validation[:100]")
test_raw = load_dataset("openslr/librispeech_asr", "clean", split="test[:100]")


train_raw = train_raw.shuffle(seed=42)

print("Raw Train:", len(train_raw))
print("Raw Val  :", len(val_raw))
print("Raw Test :", len(test_raw))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

clean/test/0000.parquet:   0%|          | 0.00/350M [00:00<?, ?B/s]

...

clean/train.360/0017.parquet:   0%|          | 0.00/463M [00:00<?, ?B/s]

clean/train.360/0018.parquet:   0%|          | 0.00/479M [00:00<?, ?B/s]

clean/train.360/0019.parquet:   0%|          | 0.00/456M [00:00<?, ?B/s]

clean/train.360/0020.parquet:   0%|          | 0.00/511M [00:00<?, ?B/s]

clean/train.360/0021.parquet:   0%|          | 0.00/491M [00:00<?, ?B/s]

_Inspect one raw audio sample_

In [4]:
def get_audio_array_and_rate(audio):
    if hasattr(audio, "get_all_samples"):
        samples = audio.get_all_samples()
        array = samples.data.squeeze(0).numpy()
        sampling_rate = samples.sample_rate
    else:
        array = audio["array"]
        sampling_rate = audio["sampling_rate"]
    return array, sampling_rate


sample = train_raw[0]
audio_array, sampling_rate = get_audio_array_and_rate(sample["audio"])


print("Audio length:", len(audio_array))
print("Sampling rate:", sampling_rate)
print("Text:", sample["text"])
print("First 20 samples:", audio_array[:20])

Audio length: 213760
Sampling rate: 16000
Text: NO PERSON MADE ANY ATTEMPT AT ACCOUNTING FOR HIS OPINION THAT HE WAS UNIQUE APPEARED SO UNDENIABLE THAT IT WAS DEEMED IMPERTINENT TO INQUIRE WHEREIN THE UNIQUITY CONSISTED
First 20 samples: [ 0.00350952  0.00134277 -0.00057983 -0.00216675 -0.00311279 -0.00369263
 -0.00421143 -0.00527954 -0.00637817 -0.00860596 -0.0116272  -0.01412964
 -0.01550293 -0.015625   -0.01516724 -0.01425171 -0.01409912 -0.01541138
 -0.01644897 -0.01773071]


### Load Whisper processor

In [5]:
MODEL_NAME = "openai/whisper-tiny"
LANGUAGE = "english"
TASK = "transcribe"
MAX_LABEL_LENGTH = 128


processor = WhisperProcessor.from_pretrained(MODEL_NAME)
processor.tokenizer.set_prefix_tokens(language=LANGUAGE, task=TASK)


print("Target sampling rate:", processor.feature_extractor.sampling_rate)
print("Language:", LANGUAGE)
print("Task:", TASK)

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Target sampling rate: 16000
Language: english
Task: transcribe


### Resample audio column for Whisper

In [7]:
target_sampling_rate = processor.feature_extractor.sampling_rate

train_raw = train_raw.cast_column("audio", Audio(sampling_rate=target_sampling_rate))
val_raw = val_raw.cast_column("audio", Audio(sampling_rate=target_sampling_rate))
test_raw = test_raw.cast_column("audio", Audio(sampling_rate=target_sampling_rate))

sample = train_raw[1]
audio_array, sampling_rate = get_audio_array_and_rate(sample["audio"])

print("New sampling rate:", sampling_rate)
print("Audio length after cast:", len(audio_array))

New sampling rate: 16000
Audio length after cast: 179760


### Build preprocessing function

In [8]:
def preprocess(example):
    audio_array, sampling_rate = get_audio_array_and_rate(example["audio"])
    
    input_features = processor.feature_extractor(
        audio_array,
        sampling_rate=sampling_rate
    ).input_features[0]
    
    labels = processor.tokenizer(
        example["text"].lower().strip(),
        truncation=True,
        max_length=MAX_LABEL_LENGTH
    ).input_ids
    
    return {
        "input_features" : input_features,
        "labels" : labels
    }

_Apply preprocessing to datasets_

In [9]:
train_dataset = train_raw.map(preprocess, remove_columns=train_raw.column_names)
val_dataset = val_raw.map(preprocess, remove_columns=val_raw.column_names)
test_dataset = test_raw.map(preprocess, remove_columns=test_raw.column_names)

print(train_dataset)
print("Input feature shape:", np.array(train_dataset[0]["input_features"]).shape)
print("Label length:", len(train_dataset[0]["labels"]))

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset({
    features: ['input_features', 'labels'],
    num_rows: 1000
})
Input feature shape: (80, 3000)
Label length: 39


### Build collator and dataloaders

In [ ]:
class DataCollatorSpeechSeq2Seq:
    def __init__(self, processor):
        self.processor = processor
    
    def __call__(self, batch):
        input_features = [{"input_features": item["input_features"]} for item in batch]
        label_features = [{"input_ids": item["labels"]} for item in batch]
        
        batch_inputs = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )
        
        batch_labels = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )
        
        labels = batch_labels["input_ids"].masked_fill(
            batch_labels["attention_mask"].ne(1),
            -100
        )
        batch_inputs["labels"] = labels

In [ ]:
data_collator = DataCollatorSpeechSeq2Seq(processor)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=data_collator)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=True, collate_fn=data_collator)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=True, collate_fn=data_collator)